# E.03 — reconstruct and verify the retained EJPS analysis

This notebook runs the same staged functions as `scripts/reproduce.py`. It uses five local, checksum-verified inputs and the retained revision targets. It performs no live ADS queries and makes no new analytical choices. Run all cells in order from a clean Python kernel.

Success means reconstructing the 55,140-record model sample, verifying both logistic models, independently checking 158,367 citation-rate records and their tie-aware percentiles, matching all eight supplied CSV tables, and exporting the three manuscript plots as STIX PDF figures.

Use ./run.sh --notebook for the full route or ./run.sh --notebook --analysis-only to skip graphics dependencies. New results go to reproduced/. Submitted artwork and normalized reference PDFs remain separate.

## 1. Locate this independent workspace and verify inputs

In [ ]:
from pathlib import Path
import sys, os, json, subprocess
start=Path.cwd().resolve()
root=next((p for p in [start,*start.parents] if (p/'config/workspace.json').is_file()),None)
if root is None: raise FileNotFoundError('Start this notebook inside its computational package.')
(root/'verification').mkdir(exist_ok=True)
cache=root/'.runtime/matplotlib';cache.mkdir(parents=True,exist_ok=True)
os.environ['MPLCONFIGDIR']=str(cache)
sys.path.insert(0,str(root/'scripts'))
from modules import analysis, plotting
state=analysis.initialize(root)
print('Verified five local input hashes:')
for name,checksum in state['checked_inputs'].items(): print(name,checksum)

## 2. Reconstruct the eligible co-citation sample

The linked-reference graph supplies seven recoverable early references. These are a subset of the review’s bibliography. Eligibility, primary-field selection, date bins, and linked-reference counts follow the retained revision.

In [ ]:
analysis.build_model_sample(state)
print('Eligible model records:',len(state['cohort']))
print('Model-sample review citers:',int(state['cohort'].cites_bh.sum()))
print('Recoverable early references:',len(state['early']))
assert len(state['cohort'])==55140
state['cohort'].groupby('cites_bh')[['bh_early','other_early']].agg(['sum','mean'])

## 3. Fit the two retained models and stratum predictions

These are adjusted associations. The second model does not test a difference between the two coefficients.

In [ ]:
analysis.fit_models_and_strata(state)
print(json.dumps(state['results'],indent=2))
state['strata'][['year_bin','arxiv_category','n_bh','n_nonbh','difference']]

## 4. Verify annual citation rates and percentile ranks

Rebuild each rate using the inclusive publication-year to latest-metric-year convention. Percentiles use average ranks for ties and all retained rate records as the denominator.

In [ ]:
analysis.verify_citation_rates(state)
assert len(state['rates'])==158367
print('Verified rate records:',len(state['rate_check']))
print('Positive-rate records:',len(state['pos']))
print('Review annual rate:',float(state['target'].avg_citations_per_year))
print('Review percentile:',float(state['target'].percentile))

## 5. Write field counts, comparisons, and the complete summary

The four displayed fields contain 537 review citers. The classified total is 551; seven nuclear-physics and seven quantum-physics records lie outside that display.

In [ ]:
analysis.write_counts_and_summary(state)
assert int(state['circulation'].sum())==551
print(state['circulation'].to_string())
print('Graph review citers:',len(state['citers']))
state['comparators'][['bibcode','avg_citations_per_year','percentile']]

## 6. Check against the independent retained revision targets

In [ ]:
check=subprocess.run([sys.executable,str(root/'scripts/check_results.py'),'--reproduced','reproduced','--report','verification/results.json'],cwd=root,capture_output=True,text=True)
if check.returncode: raise RuntimeError(check.stdout+check.stderr)
report=json.loads((root/'verification/results.json').read_text())
assert report['passed'] and len(report['rebuilt_tables_compared'])==8
print('Both models, rate identities, summary, and all eight CSV comparisons passed.')
print('Compared tables:',*report['rebuilt_tables_compared'],sep='\n')

## 7. Export manuscript figures and record the checked run

Plot text uses bundled STIX Two Text. The impact PDF is 156 × 90 mm and the waffle PDF is 156 × 122.218 mm, retaining the author’s 7.25:5.68 aspect ratio; the top waffle panel has no coordinate axes. The stratum plot follows the author’s supplied ggplot specification, including 8.5-point x-axis labels, and exports through native Quartz at exactly 7 × 4 inches (177.8 × 101.6 mm). The manuscript includes this PDF at 156 mm width, preserving its aspect ratio. The historical facsimile is handled separately in the manuscript source. See the figure registry and NORMALIZATION.md for the distinction between retained author artwork and normalized displays.

In [ ]:
if os.environ.get('DAMADI_ANALYSIS_ONLY') == '1':
    print('Numerical-only mode: approved PDF export skipped.')
else:
    figures=plotting.export_all(root)
    for figure in figures: print(figure['pdf'],figure['width_mm'],figure['height_mm'])
record=analysis.write_run_record(state,verification=report)
assert record['verification']['passed']
print('Checked reconstruction complete. Supplied evidence remains unchanged.')


## Interpretation and limits

Citation overlap does not establish causal influence, agreement, or interpretive adherence. Linked references are incomplete relative to full bibliographies. The original ADS retrieval timestamp is not embedded in the retained tables. The named historical comparators are not an exhaustive genre population. This run verifies the retained calculation; it does not add an independent empirical validation sample.